# Varhadi voice: fine-tune MMS-TTS Marathi on narrator clips (Luke) - Kaggle version
PRIVATE RESEARCH USE ONLY. The recordings have an unverified licence and carry a real narrator's voice; never publish the result or make the dataset public. Settings: accelerator GPU (P100 or T4), Internet ON, dataset `varhadi-luke-tts-private` attached. Use **Save Version -> Save & Run All (Commit)**: it runs on Kaggle's servers (about 75 min) even if this tab is closed. Output: `varhadi_voice.zip` (the generator only, about 135 MB). Recipe: github.com/ylacombe/finetune-hf-vits (MMS checkpoints are CC BY-NC 4.0).

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/ylacombe/finetune-hf-vits.git /kaggle/working/finetune-hf-vits
%cd /kaggle/working/finetune-hf-vits
!git checkout -q 6f3f51f4d667f5c3eef89484d151ffd39d2c2b89
!pip -q install -r requirements.txt
# the 2024 repo needs the transformers 4.x VitsConfig (v5 dropped pad_token_id)
!pip -q install "transformers==4.46.3"
# the script reads batch["speaker_id"] even for a single-speaker dataset
!sed -i 's/speaker_id=batch\["speaker_id"\]/speaker_id=batch.get("speaker_id")/' run_vits_finetuning.py
# matplotlib 3.10 removed tostring_rgb (only used to draw alignment plots at eval time)
!sed -i 's/np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)/np.ascontiguousarray(np.asarray(fig.canvas.buffer_rgba())[..., :3]).reshape(-1)/' utils/plot.py
!cd monotonic_align && mkdir -p monotonic_align && python setup.py build_ext --inplace

In [ ]:
import glob, os, shutil
os.makedirs('/kaggle/working/luke_ds/wavs', exist_ok=True)
meta = glob.glob('/kaggle/input/**/luk_part0/metadata.csv', recursive=True)[0]
root = os.path.dirname(os.path.dirname(meta))            # Kaggle unpacked each zip into its own luk_partN folder
for f in glob.glob(root + '/luk_part*/wavs/*'):
    shutil.copy(f, '/kaggle/working/luke_ds/wavs/')
shutil.copy(meta, '/kaggle/working/luke_ds/metadata.csv')
print(len(glob.glob('/kaggle/working/luke_ds/wavs/*')), 'clips;', open('/kaggle/working/luke_ds/metadata.csv', encoding='utf-8').read()[:300])

In [ ]:
%cd /kaggle/working/finetune-hf-vits
!python convert_original_discriminator_checkpoint.py --language_code mar --pytorch_dump_folder_path /kaggle/working/mms-tts-mar-train
!ls -la /kaggle/working/mms-tts-mar-train

In [ ]:
import json
true, false = True, False
cfg = {
 "project_name": "varhadi_tts_luke",
 "push_to_hub": false,
 "output_dir": "/kaggle/working/vits_luke",
 "model_name_or_path": "/kaggle/working/mms-tts-mar-train",
 "dataset_name": "/kaggle/working/luke_ds",
 "audio_column_name": "audio",
 "text_column_name": "text",
 "train_split_name": "train",
 "eval_split_name": "train",
 "max_eval_samples": 12,
 "max_duration_in_seconds": 20,
 "min_duration_in_seconds": 1.0,
 "max_tokens_length": 500,
 "do_lower_case": false,
 "preprocessing_num_workers": 1,
 "full_generation_sample_text": "तुमी पिकाले पाणी देऊन झालं का? काई अडचण असीन तर कृषी सहाय्यकाले सांगा.",
 "do_train": true,
 "do_eval": true,
 "num_train_epochs": 100,
 "per_device_train_batch_size": 8,
 "per_device_eval_batch_size": 8,
 "learning_rate": 2e-05,
 "adam_beta1": 0.8,
 "adam_beta2": 0.99,
 "fp16": true,
 "seed": 456,
 "eval_steps": 500,
 "save_steps": 500,
 "save_total_limit": 2,
 "logging_steps": 50,
 "report_to": "none",
 "weight_disc": 3,
 "weight_fmaps": 1,
 "weight_gen": 1,
 "weight_kl": 1.5,
 "weight_duration": 1,
 "weight_mel": 35
}
json.dump(cfg, open('/kaggle/working/finetune_luke.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('config written')

In [ ]:
%cd /kaggle/working/finetune-hf-vits
# ONE GPU only: on T4 x2 accelerate started 2 processes and a child crashed at ~step 2200 (ChildFailedError). ~1 step/s, 4400 steps.
# The full log goes to train_full.log; the screen shows every 100th step and, at the end, the last non-spam lines (errors included).
!rm -rf /kaggle/working/vits_luke
!CUDA_VISIBLE_DEVICES=0 accelerate launch --num_processes 1 --num_machines 1 run_vits_finetuning.py /kaggle/working/finetune_luke.json 2>&1 | tr '\r' '\n' | tee /kaggle/working/train_full.log | grep --line-buffered -E "Steps: .*\| [0-9]*00/4400" | cut -c1-150
!grep -v -E "BufferError|^Traceback \(most recent call last\):$" /kaggle/working/train_full.log | tail -30 | cut -c1-220
!ls /kaggle/working/vits_luke

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
m, info = VitsModel.from_pretrained('/kaggle/working/vits_luke', output_loading_info=True)   # drops the training-only discriminator
print('missing keys:', info['missing_keys'], '| dropped:', len(info['unexpected_keys']))
tok = AutoTokenizer.from_pretrained('/kaggle/working/vits_luke'); m.eval(); torch.manual_seed(0)
with torch.no_grad(): w = m(**tok('तुमी पिकाले पाणी देऊन झालं का', return_tensors='pt')).waveform[0]
print('test sentence: %.2f s, rms %.3f' % (w.shape[0] / m.config.sampling_rate, float(w.pow(2).mean().sqrt())))
m.save_pretrained('/kaggle/working/varhadi_voice'); tok.save_pretrained('/kaggle/working/varhadi_voice')
!cd /kaggle/working/varhadi_voice && zip -q -r /kaggle/working/varhadi_voice.zip .
# keep only the zip in the notebook output (checkpoints and a copy of the data would clutter it)
!cd /kaggle/working && rm -rf finetune-hf-vits luke_ds vits_luke mms-tts-mar-train varhadi_voice finetune_luke.json
!ls -la /kaggle/working